Setup

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import torch.optim as optim
import time
import json

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = torchvision.datasets.Flowers102(root='./data', split='train', download=True, transform=train_transform)
val_dataset = torchvision.datasets.Flowers102(root='./data', split='val', download=True, transform=val_transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

print(len(train_dataset), "train,", len(val_dataset), "val")

100%|██████████| 345M/345M [00:23<00:00, 14.9MB/s]
100%|██████████| 502/502 [00:00<00:00, 1.94MB/s]
100%|██████████| 15.0k/15.0k [00:00<00:00, 44.0MB/s]


1020 train, 1020 val


Model builder and training loop

In [2]:
def build_model(setting):
    if setting == "scratch_full":
        model = models.resnet152(weights=None)
    else:
        model = models.resnet152(weights=models.ResNet152_Weights.IMAGENET1K_V1)

    model.fc = nn.Linear(2048, 102)

    if setting == "pretrained_head":
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("fc.")
    elif setting == "pretrained_layer4":
        for name, p in model.named_parameters():
            p.requires_grad = name.startswith("fc.") or name.startswith("layer4.")
    else:
        for p in model.parameters():
            p.requires_grad = True

    return model.to(device)


def run_setting(setting, lr, num_epochs=15):
    model = build_model(setting)
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"\n=== {setting} | {sum(p.numel() for p in params)/1e6:.1f}M trainable ===")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {"train_acc": [], "val_acc": [], "epoch_time": []}

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        correct, total = 0, 0
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            optimizer.zero_grad()
            out = model(images)
            loss = criterion(out, targets)
            loss.backward()
            optimizer.step()
            correct += out.argmax(1).eq(targets).sum().item()
            total += targets.size(0)
        scheduler.step()
        train_acc = 100 * correct / total

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, targets in val_loader:
                images, targets = images.to(device), targets.to(device)
                out = model(images)
                correct += out.argmax(1).eq(targets).sum().item()
                total += targets.size(0)
        val_acc = 100 * correct / total
        dt = time.time() - t0

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["epoch_time"].append(dt)
        print(f"epoch {epoch+1}: train {train_acc:.2f}%  val {val_acc:.2f}%  ({dt:.0f}s)")

    return history

Running

In [3]:
results = {}
results["pretrained_head"] = run_setting("pretrained_head", lr=1e-3)
results["pretrained_layer4"] = run_setting("pretrained_layer4", lr=1e-4)
results["pretrained_full"] = run_setting("pretrained_full", lr=1e-4)
results["scratch_full"] = run_setting("scratch_full", lr=1e-3)

with open("task1_4_results.json", "w") as f:
    json.dump(results, f, indent=2)

Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth


100%|██████████| 230M/230M [00:01<00:00, 194MB/s]



=== pretrained_head | 0.2M trainable ===
epoch 1: train 9.31%  val 43.53%  (29s)
epoch 2: train 60.49%  val 67.55%  (26s)
epoch 3: train 82.45%  val 77.06%  (27s)
epoch 4: train 92.55%  val 80.29%  (27s)
epoch 5: train 93.63%  val 84.22%  (28s)
epoch 6: train 96.47%  val 84.31%  (28s)
epoch 7: train 98.43%  val 86.08%  (28s)
epoch 8: train 98.63%  val 87.16%  (28s)
epoch 9: train 99.02%  val 89.02%  (27s)
epoch 10: train 99.31%  val 88.92%  (28s)
epoch 11: train 99.80%  val 88.82%  (28s)
epoch 12: train 99.51%  val 89.02%  (28s)
epoch 13: train 99.90%  val 89.41%  (28s)
epoch 14: train 99.71%  val 89.02%  (28s)
epoch 15: train 99.80%  val 88.82%  (28s)

=== pretrained_layer4 | 15.2M trainable ===
epoch 1: train 20.78%  val 60.69%  (29s)
epoch 2: train 79.22%  val 79.22%  (29s)
epoch 3: train 90.20%  val 86.47%  (29s)
epoch 4: train 96.27%  val 91.27%  (29s)
epoch 5: train 98.92%  val 92.55%  (29s)
epoch 6: train 99.80%  val 92.94%  (30s)
epoch 7: train 100.00%  val 93.63%  (29s)
epoch

Summary table

In [4]:
print(f"{'setting':22s} {'best val':>9s} {'total time':>11s}")
for name, h in results.items():
    print(f"{name:22s} {max(h['val_acc']):8.2f}% {sum(h['epoch_time']):10.0f}s")

setting                 best val  total time
pretrained_head           89.41%        415s
pretrained_layer4         94.12%        438s
pretrained_full           94.12%        687s
scratch_full              13.82%        691s
